In [16]:
"""
MODEL LOOKS TERRIBLE – FIND THE ENCODER BUG
 • 1 000 rows, 5 colours, 5-class label
 • Train split deliberately lacks ‘cyan’ and ‘magenta’
 • BUG: pd.factorize() called separately on train and test
   (order-of-appearance mapping)  → test accuracy ≈ 0.20
 • FIX: use the mapping learned on train for the test split
"""
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model   import LogisticRegression
from sklearn.metrics        import accuracy_score

rng      = np.random.RandomState(0)
classes  = np.array(['red','green','blue','cyan','magenta'])

# ------------------------------------------------------------
# 1.  make a data set whose target is strongly tied to colour
# ------------------------------------------------------------
n = 1_000
X = pd.DataFrame({"colour": rng.choice(classes, n)})

def noisy_identity(col):
    # correct label 80 % of the time, otherwise random wrong class
    y = col.copy()
    mask = rng.rand(n) > 0.80
    for i in np.where(mask)[0]:
        y.iat[i] = rng.choice(classes[classes != y.iat[i]])
    return y

y = noisy_identity(X['colour'])



# ------------------------------------------------------------
# 2.  train/test split WITHOUT shuffling
#     first 700 rows = train  (only red/green/blue)
#     last 300 rows  = test   (contains cyan & magenta)
# ------------------------------------------------------------
X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.30, shuffle=False)

# sanity-check the category sets
print("Train categories:", X_tr['colour'].unique())
print("Test  categories:", X_te['colour'].unique(), "\n")
X



Train categories: ['magenta' 'red' 'cyan' 'green' 'blue']
Test  categories: ['red' 'blue' 'green' 'magenta' 'cyan'] 



,colour
0,magenta
1,red
2,cyan
3,cyan
4,cyan
...,...
995,magenta
996,cyan
997,green
998,green


## 📊 COMPLETE SUMMARY - The Whole Picture

```
STEP 1: Create Data
─────────────────────────────────────────────────
INPUT (X)         OUTPUT (y)         Relationship
─────────────────────────────────────────────────
'red'         →   'red'              ✓ Match (80%)
'blue'        →   'blue'             ✓ Match
'green'       →   'cyan'             ✗ Noise (20%)
'magenta'     →   'red'              ✗ Noise
... 1000 rows total ...


STEP 2: Split into Train/Test
─────────────────────────────────────────────────
TRAIN (700 rows)          TEST (300 rows)
─────────────────────────────────────────────────
INPUT: colour             INPUT: colour
OUTPUT: colour label      OUTPUT: colour label


STEP 3a: BUGGY ENCODING ❌
─────────────────────────────────────────────────
TRAIN: pd.factorize()     TEST: pd.factorize()
magenta → 0               red → 0      ← MISMATCH!
red → 1                   blue → 1     ← MISMATCH!
cyan → 2                  green → 2    ← MISMATCH!

Model learns: "0 means magenta"
At test: We give it 0, but it's actually "red"!
Result: Model is completely confused → 20% accuracy


STEP 3b: CORRECT ENCODING ✓
─────────────────────────────────────────────────
TRAIN: pd.factorize()     TEST: Use TRAIN mapping
magenta → 0               magenta → 0  ← Same!
red → 1                   red → 1      ← Same!
cyan → 2                  cyan → 2     ← Same!

Model learns: "0 means magenta"
At test: We give it 0 for "magenta"
Result: Model works correctly → 80% accuracy
```

**The Lesson:** Always use the SAME encoding for both train and test!

## ⚠️ CRITICAL RULE: Train and Test Must Use the SAME Mapping

### **Do we 100% of the time have to use the same mapping?**

**YES - ABSOLUTELY! This is a fundamental rule of machine learning.**

### **Why is this non-negotiable?**

The model learns a **specific meaning** for each number during training:

```
Training phase:
  red → 0      Model learns: "When I see 0, predict 'red'"
  blue → 1     Model learns: "When I see 1, predict 'blue'"
  green → 2    Model learns: "When I see 2, predict 'green'"
```

If you change the mapping at test time, you're speaking a **different language**:

```
❌ WRONG - Different mapping at test:
  blue → 0     You give the model 0
  Model thinks: "0 means red!"
  Model predicts: "red"
  Actual answer: "blue"
  Result: WRONG!
```

### **Real-world analogy:**

Imagine you teach a child:
- 🍎 = 1 means "apple"
- 🍌 = 2 means "banana"
- 🍊 = 3 means "orange"

Then at test time, you change the rules:
- Show them 🍌 but call it "1"
- The child says "apple" (because they learned 1=apple)
- You mark them wrong because you meant "banana"

**Who made the mistake? YOU did!** The child learned correctly, but you changed the encoding.

### **This applies to ALL encoding methods:**

1. **Label Encoding:** Same mapping for categories
2. **One-Hot Encoding:** Same column order
3. **Feature Scaling:** Same mean/std from training
4. **Any transformation:** Same parameters from training

### **The Golden Rule:**

```python
# ✅ ALWAYS do this:
# 1. FIT on training data only
encoder.fit(X_train)

# 2. TRANSFORM both train and test with those same parameters
X_train_enc = encoder.transform(X_train)
X_test_enc = encoder.transform(X_test)  # Uses SAME mapping!

# ❌ NEVER do this:
encoder.fit(X_train)
X_train_enc = encoder.transform(X_train)

encoder.fit(X_test)  # ← WRONG! Different mapping!
X_test_enc = encoder.transform(X_test)
```

### **What about production/new data?**

In production, you use the **SAME mapping forever**:

```python
# Training time (Month 1):
encoder.fit(training_data)
save_model(encoder)  # Save the mapping!

# Production time (Month 2, 3, 4...):
encoder = load_model()  # Load the SAME mapping
new_data_encoded = encoder.transform(new_data)  # Always same!
```

**Bottom line: The mapping is part of your model. It NEVER changes after training!** 🔒

## Understanding the `noisy_identity()` function

This function creates a **classification task** where:
- **80% of the time**: The label matches the colour perfectly
- **20% of the time**: The label is randomly wrong (noise)

This simulates a real-world scenario where the feature strongly predicts the target, but not perfectly.

In [17]:
# Let's visualize what noisy_identity() does with a small example
print("="*70)
print("DEMONSTRATING noisy_identity() - Creating Labels with 20% Noise")
print("="*70)

# Create a comparison DataFrame
comparison = pd.DataFrame({
    'Feature (colour)': X['colour'],
    'Label (y)': y,
    'Match?': X['colour'] == y
})

# Show first 20 rows to see the pattern
print("\n📊 First 20 rows showing feature vs label:")
print(comparison.head(20).to_string(index=True))

# Calculate statistics
total_rows = len(comparison)
matches = (comparison['Match?'] == True).sum()
mismatches = (comparison['Match?'] == False).sum()
match_percentage = (matches / total_rows) * 100

print("\n" + "="*70)
print("📈 STATISTICS:")
print("="*70)
print(f"Total rows:        {total_rows}")
print(f"Matches (✓):       {matches} ({match_percentage:.1f}%)")
print(f"Mismatches (✗):    {mismatches} ({100-match_percentage:.1f}%)")

print("\n💡 WHAT THIS MEANS:")
print("   • 80% of rows: colour and label are IDENTICAL (perfect prediction)")
print("   • 20% of rows: label is a DIFFERENT random colour (noise)")
print("   • This creates a realistic ML task: strong signal + some noise")

# Show some mismatch examples
print("\n🔍 Examples where label DOESN'T match feature (the 20% noise):")
mismatches_df = comparison[comparison['Match?'] == False].head(10)
print(mismatches_df.to_string(index=True))

DEMONSTRATING noisy_identity() - Creating Labels with 20% Noise

📊 First 20 rows showing feature vs label:
   Feature (colour) Label (y)  Match?
0           magenta   magenta    True
1               red       red    True
2              cyan      cyan    True
3              cyan      cyan    True
4              cyan      cyan    True
5             green     green    True
6              cyan       red   False
7              blue   magenta   False
8           magenta   magenta    True
9               red       red    True
10              red       red    True
11          magenta   magenta    True
12             blue      blue    True
13            green     green    True
14              red       red    True
15            green     green    True
16            green      blue   False
17              red       red    True
18            green      cyan   False
19          magenta   magenta    True

📈 STATISTICS:
Total rows:        1000
Matches (✓):       775 (77.5%)
Mismatches (✗):    225 (2

In [18]:
# ------------------------------------------------------------
# 3.  BUG – encode each split independently  (order matters!)
# ------------------------------------------------------------
def encode_series(s):
    codes, uniques = pd.factorize(s, sort=False)
    return codes, uniques

X_tr_enc, uniq_tr = encode_series(X_tr['colour'])      # fit here
X_te_enc, _       = encode_series(X_te['colour'])      # ❌ re-fit here

clf = LogisticRegression(max_iter=400,
                         multi_class='multinomial').fit(
                         X_tr_enc.reshape(-1,1), y_tr)

print("Accuracy WITH bug :", round(
        accuracy_score(y_te, clf.predict(X_te_enc.reshape(-1,1))), 3))
print(uniq_tr, "\n")
print(_)


Accuracy WITH bug : 0.05
Index(['magenta', 'red', 'cyan', 'green', 'blue'], dtype='object') 

Index(['red', 'blue', 'green', 'magenta', 'cyan'], dtype='object')


/home/edward/github/data_prep_and_feature_engineering/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [19]:


# ------------------------------------------------------------
# 4.  FIX – transform test split with the train mapping only
# ------------------------------------------------------------
mapping = {c: i for i, c in enumerate(uniq_tr)}
X_te_fixed = X_te['colour'].map(mapping).fillna(-1).astype(int)

print("Accuracy after fix:", round(
        accuracy_score(y_te, clf.predict(X_te_fixed.values.reshape(-1,1))), 3))

Accuracy after fix: 0.62


# Model too bad Example 2

**Bug:** Features and target are shuffled independently, breaking their correspondence.

**Result:** Model accuracy ≈ random guessing (0.20)

In [20]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Create a simple dataset where feature strongly predicts target
rng = np.random.RandomState(42)
n = 2000

# Feature: age groups
X = pd.DataFrame({
    'age': rng.randint(18, 80, n),
    'score': rng.rand(n) * 100
})

# Target: age category (strongly correlated with age)
def get_category(age):
    if age < 30: return 'young'
    elif age < 45: return 'adult'
    elif age < 60: return 'middle'
    else: return 'senior'

y = X['age'].apply(get_category)        # this defines the field we want to predict

print("Original correlation check:")
print(pd.crosstab(X['age'] // 15, y))

Original correlation check:
age  adult  middle  senior  young
age                              
1        0       0       0    375
2      466       0       0      0
3        0     496       0      0
4        0       0     482      0
5        0       0     181      0


In [21]:
# ❌ BUG: Shuffle features and target independently!
X_shuffled = X.sample(frac=1, random_state=98).reset_index(drop=True)
y_shuffled = y.sample(frac=1, random_state=77).reset_index(drop=True)  # Different seed! -> this is where we shuffle the field we want to predict, 
#if use different random state, then the label mapping will be wrong all over the dataset
"""
# Result after combining:
    features_from  label_from   WRONG!
0   Carol (65)     Bob (adult)  ← 65yo labeled as "adult"
1   Alice (25)     Eve (middle) ← 25yo labeled as "middle"
2   Eve (50)       Carol (senior) ← 50yo labeled as "senior"
3   David (30)     Alice (young) ← 30yo labeled as "young" (lucky!)
4   Bob (45)       David (young) ← 45yo labeled as "young"
"""



'\n# Result after combining:\n    features_from  label_from   WRONG!\n0   Carol (65)     Bob (adult)  ← 65yo labeled as "adult"\n1   Alice (25)     Eve (middle) ← 25yo labeled as "middle"\n2   Eve (50)       Carol (senior) ← 50yo labeled as "senior"\n3   David (30)     Alice (young) ← 30yo labeled as "young" (lucky!)\n4   Bob (45)       David (young) ← 45yo labeled as "young"\n'

In [ ]:
# Now features don't match their labels
X_train, X_test, y_train, y_test = train_test_split(
    X_shuffled, y_shuffled, test_size=0.25, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print("\n❌ WITH BUG (shuffled independently):")
print(f"Train accuracy: {model.score(X_train, y_train):.3f}")
print(f"Test accuracy: {model.score(X_test, y_test):.3f}")
print(f"Random baseline: {1/len(y.unique()):.3f}")

In [ ]:
# ✓ FIX: Don't shuffle independently, or shuffle together
# Method 1: Don't shuffle at all if indices already align
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Method 2: If you must shuffle, shuffle together
# combined = pd.concat([X.reset_index(drop=True), y.reset_index(drop=True)], axis=1)
# combined = combined.sample(frac=1, random_state=42).reset_index(drop=True)

model_fixed = RandomForestClassifier(n_estimators=100, random_state=42)
model_fixed.fit(X_train, y_train)

print("\n✓ AFTER FIX:")
print(f"Train accuracy: {model_fixed.score(X_train, y_train):.3f}")
print(f"Test accuracy: {model_fixed.score(X_test, y_test):.3f}")